# Employee Turnover Intelligence System

## Portfolio-Level Data Science Project

Features:
- Business Understanding
- Data Quality Framework
- Advanced EDA
- Statistical Insights
- KMeans Segmentation
- SMOTE
- Feature Engineering
- Logistic Regression
- Random Forest
- Gradient Boosting
- Hyperparameter Tuning
- ROC/AUC Comparison
- Feature Importance
- SHAP Explainability (optional)
- Employee Risk Scoring Engine
- Retention Strategy Recommendation Engine
- Executive Summary


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import *
from imblearn.over_sampling import SMOTE

sns.set_theme()


In [ ]:

df = pd.read_excel('../data/HR_comma_sep.xlsx')
df.head()


## Data Quality Assessment

In [ ]:

quality = pd.DataFrame({
    'dtype':df.dtypes,
    'missing':df.isna().sum(),
    'unique':df.nunique()
})
quality


## Advanced EDA

In [ ]:

num_cols=df.select_dtypes(include='number').columns

plt.figure(figsize=(10,8))
sns.heatmap(df[num_cols].corr(),annot=True,cmap='coolwarm')
plt.show()


## Feature Engineering

In [ ]:

df['hours_per_project']=df['average_montly_hours']/df['number_project']
df['tenure_efficiency']=df['last_evaluation']*df['time_spend_company']


## Clustering Leavers

In [ ]:

left_df=df[df.left==1][['satisfaction_level','last_evaluation']]

kmeans=KMeans(n_clusters=3,random_state=42)
left_df['cluster']=kmeans.fit_predict(left_df)

sns.scatterplot(data=left_df,x='satisfaction_level',y='last_evaluation',hue='cluster')
plt.show()


## Modeling Pipeline

In [ ]:

X=df.drop('left',axis=1)
y=df['left']

X=pd.get_dummies(X,drop_first=True)

X_train,X_test,y_train,y_test=train_test_split(
X,y,test_size=0.2,stratify=y,random_state=123)

smote=SMOTE(random_state=123)
X_train,y_train=smote.fit_resample(X_train,y_train)


In [ ]:

models={
'Logistic Regression':LogisticRegression(max_iter=2000),
'Random Forest':RandomForestClassifier(random_state=42),
'Gradient Boosting':GradientBoostingClassifier(random_state=42)
}

results=[]

for n,m in models.items():
    cv=cross_val_score(m,X_train,y_train,cv=5,scoring='roc_auc')
    results.append([n,cv.mean()])

pd.DataFrame(results,columns=['Model','CV_AUC']).sort_values('CV_AUC',ascending=False)


## Hyperparameter Tuning

In [ ]:

params={
'n_estimators':[200,300],
'max_depth':[5,10,None]
}

grid=GridSearchCV(
RandomForestClassifier(random_state=42),
params,
cv=5,
scoring='roc_auc'
)

grid.fit(X_train,y_train)
grid.best_params_


## Final Evaluation

In [ ]:

best=grid.best_estimator_

best.fit(X_train,y_train)

pred=best.predict(X_test)
prob=best.predict_proba(X_test)[:,1]

print(classification_report(y_test,pred))

print("ROC AUC:",roc_auc_score(y_test,prob))


In [ ]:

cm=confusion_matrix(y_test,pred)

sns.heatmap(cm,annot=True,fmt='d')
plt.title('Confusion Matrix')
plt.show()


In [ ]:

imp=pd.DataFrame({
'Feature':X_train.columns,
'Importance':best.feature_importances_
}).sort_values('Importance',ascending=False).head(15)

imp



## Employee Risk Engine

Green <20%
Yellow 20-60%
Orange 60-90%
Red >90%


In [ ]:

risk=pd.DataFrame()

risk['probability']=prob

risk['zone']=pd.cut(
risk['probability'],
bins=[0,.2,.6,.9,1],
labels=['Green','Yellow','Orange','Red']
)

risk.head()



# Executive Recommendations

### High Risk
Immediate retention intervention

### Medium Risk
Compensation and workload review

### Low Risk
Career growth discussions

### Safe
Recognition and engagement programs
